In [0]:
# --------------------------------------------
# 1. Create Sample Data
# --------------------------------------------

data = [
    (1, "John", "IT", 50000),
    (2, "Sara", "HR", 45000),
    (3, "Mike", "Finance", 60000)
]

columns = ["id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)

df.show()

In [0]:
# --------------------------------------------
# 2. Write Data as Delta Table
# --------------------------------------------

path = "/Volumes/workspace/data_files/csv_files/sample.delta"

df.write.format("delta").mode("overwrite").save(path)

employees = spark.read.format("delta").load(path)
employees.show()

In [0]:
# --------------------------------------------
# 3. Delta Table Object
# --------------------------------------------

from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, path)

In [0]:
# --------------------------------------------
# 4. UPDATE Operation (ACID Transaction)
# --------------------------------------------

delta_table.update(
    condition = "id = 1",
    set = {"salary": "55000"}
)

spark.read.format("delta").load(path).show()

In [0]:
# --------------------------------------------
# 5. DELETE Operation
# --------------------------------------------

delta_table.delete("id = 2")

spark.read.format("delta").load(path).show()

In [0]:
# --------------------------------------------
# 6. View Delta Table History
# --------------------------------------------

delta_table.history().show(truncate=False)

In [0]:
# --------------------------------------------
# 7. TIME TRAVEL
# --------------------------------------------

# Read older version of table

old_version = spark.read.format("delta") \
.option("versionAsOf", 0) \
.load(path)

old_version.show()

In [0]:
# --------------------------------------------
# 8. MERGE (UPSERT)
# --------------------------------------------

new_data = [
    (1, "John", "IT", 70000),
    (4, "David", "Marketing", 48000)
]

new_df = spark.createDataFrame(new_data, columns)

delta_table.alias("old").merge(
    new_df.alias("new"),
    "old.id = new.id"
).whenMatchedUpdate(
    set = {"salary": "new.salary"}
).whenNotMatchedInsertAll().execute()

spark.read.format("delta").load(path).show()

In [0]:
# --------------------------------------------
# 8. MERGE (UPSERT)
# --------------------------------------------

new_data = [
    (1, "John", "IT", 70000),
    (4, "David", "Marketing", 48000)
]

new_df = spark.createDataFrame(new_data, columns)

delta_table.alias("old").merge(
    new_df.alias("new"),
    "old.id = new.id"
).whenMatchedUpdate(
    set = {"salary": "new.salary"}
).whenNotMatchedInsertAll().execute()

spark.read.format("delta").load(path).show()

In [0]:
# --------------------------------------------
# 9. SCHEMA EVOLUTION
# --------------------------------------------

data2 = [
    (5, "Emma", "Sales", 52000, "India")
]

columns2 = ["id","name","department","salary","country"]

df2 = spark.createDataFrame(data2, columns2)

df2.write.format("delta") \
.mode("append") \
.option("mergeSchema","true") \
.save(path)

spark.read.format("delta").load(path).printSchema()

In [0]:
# --------------------------------------------
# 11. ZORDER (Query Optimization)
# --------------------------------------------

spark.sql(f"OPTIMIZE delta.`{path}` ZORDER BY (department)")
# --------------------------------------------
# 12. VACUUM (Clean Old Files)
# --------------------------------------------

delta_table.vacuum()